# AE Reconstruction Comparison

Notebook wrapper for the MLP vs atlas-free CNN autoencoder reconstruction comparison. The evaluation code lives in `atlas_free_cnn.evaluation.compare_ae_reconstruction`; this notebook only configures and calls it.

In [1]:
from pathlib import Path
import sys

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "neurovlm").exists() and (candidate / "experiments" / "3dcnn" / "atlas_free_cnn").exists():
            return candidate
    raise RuntimeError("Could not find repo root. Start Jupyter from the neurovlm repo or update this cell.")

REPO_ROOT = find_repo_root()
THREEDCNN = REPO_ROOT / "experiments" / "3dcnn"
MODEL_COMPARISON_DIR = THREEDCNN / "model_comparison"
for path in [REPO_ROOT / "src", THREEDCNN, MODEL_COMPARISON_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

REPO_ROOT

PosixPath('/Users/borng/code/lab_work/neurovlm')

In [2]:
import pandas as pd

from atlas_free_cnn.evaluation.compare_ae_reconstruction import (
    AE_MODEL_IDS,
    DATASETS,
    DEFAULT_OUTPUT_DIR,
    run_comparison,
)

list(DATASETS), list(AE_MODEL_IDS)

(['pubmed', 'nilearn', 'neurovault'],
 ['mlp_neurovlm',
  'cnn_ae_mixed',
  'cnn_ae_pubmed',
  'cnn_ae_nilearn',
  'cnn_ae_neurovault'])

## Configure

Set `RUN_MODE` below: `"quick"` is a fast sanity check (PubMed only, MLP plus
one CNN model, two samples); `"full"` is the actual comparison across all
datasets/models (downloads CNN + MLP checkpoints and the unified test split
on first run, then reads from local HF cache). Everything below -- Run
Comparison, Inspect Outputs, Visualize Results -- uses whichever mode you
pick here.

In [3]:
# RUN_MODE:
#   "quick" - fast sanity check: PubMed only, MLP + one CNN model, two samples.
#   "full"  - the actual comparison across all datasets/models.
RUN_MODE = "full"

if RUN_MODE == "quick":
    DATASETS_TO_RUN = ["pubmed"]
    MODELS = ["mlp_neurovlm", "cnn_ae_mixed"]
    LIMIT = 2
elif RUN_MODE == "full":
    DATASETS_TO_RUN = ["pubmed", "neurovault", "nilearn"]
    MODELS = list(AE_MODEL_IDS)
    LIMIT = 16
else:
    raise ValueError(f"Unknown RUN_MODE {RUN_MODE!r}; expected 'quick' or 'full'")

DEVICE = "cpu"
BATCH_SIZE = 8
SKIP_VOXEL_AUROC = False
SKIP_MLP_FLAT = False

preferred_test_jsonl = REPO_ROOT / "experiments" / "3dcnn" / "atlas_free_cnn" / "cache" / "unified_jsonl" / "splits" / "test.jsonl"
TEST_JSONL = preferred_test_jsonl if preferred_test_jsonl.exists() else None

OUTPUT_DIR = REPO_ROOT / DEFAULT_OUTPUT_DIR

{
    "run_mode": RUN_MODE,
    "datasets": DATASETS_TO_RUN,
    "models": MODELS,
    "limit": LIMIT,
    "device": DEVICE,
    "test_jsonl": str(TEST_JSONL) if TEST_JSONL else None,
    "output_dir": str(OUTPUT_DIR),
}

{'run_mode': 'full',
 'datasets': ['pubmed', 'neurovault', 'nilearn'],
 'models': ['mlp_neurovlm',
  'cnn_ae_mixed',
  'cnn_ae_pubmed',
  'cnn_ae_nilearn',
  'cnn_ae_neurovault'],
 'limit': 16,
 'device': 'cpu',
 'test_jsonl': '/Users/borng/code/lab_work/neurovlm/experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl/splits/test.jsonl',
 'output_dir': '/Users/borng/code/lab_work/neurovlm/experiments/3dcnn/atlas_free_cnn/outputs/model_comparison'}

## Run Comparison

In [4]:
result = run_comparison(
    datasets=DATASETS_TO_RUN,
    models=MODELS,
    limit=LIMIT,
    device=DEVICE,
    output_dir=OUTPUT_DIR,
    test_jsonl=TEST_JSONL,
    batch_size=BATCH_SIZE,
    include_voxel_auroc=not SKIP_VOXEL_AUROC,
    include_mlp_flat=not SKIP_MLP_FLAT,
)

result

/Users/borng/code/lab_work/neurovlm/src/neurovlm/text_to_brain_metrics.py:420: RuntimeWarning: overflow encountered in exp
  p = 1 / (1 + np.exp(-logits)) # sigmoid(logits)


{'by_sample_path': PosixPath('/Users/borng/code/lab_work/neurovlm/experiments/3dcnn/atlas_free_cnn/outputs/model_comparison/ae_reconstruction_by_sample.csv'),
 'summary_csv_path': PosixPath('/Users/borng/code/lab_work/neurovlm/experiments/3dcnn/atlas_free_cnn/outputs/model_comparison/ae_reconstruction_summary.csv'),
 'summary_json_path': PosixPath('/Users/borng/code/lab_work/neurovlm/experiments/3dcnn/atlas_free_cnn/outputs/model_comparison/ae_reconstruction_summary.json'),
 'summary': [{'dataset': 'neurovault',
   'model_id': 'cnn_ae_mixed',
   'comparison_space': 'native_atlas_free_volume',
   'n_samples': 16,
   'n_supported': 16,
   'n_unsupported': 0,
   'unsupported_reasons': '',
   'mse_mean': 0.008799171727332578,
   'mse_std': 0.00480221079836505,
   'reconstruction_mse_mean': 0.008799171727332578,
   'reconstruction_mse_std': 0.00480221079836505,
   'mae_mean': 0.04524007912550587,
   'mae_std': 0.020900856801459214,
   'foreground_mse_mean': 0.026669895683880895,
   'foregro

## Inspect Outputs

In [5]:
summary = pd.read_csv(result["summary_csv_path"])
summary

,dataset,model_id,comparison_space,n_samples,n_supported,n_unsupported,unsupported_reasons,mse_mean,mse_std,reconstruction_mse_mean,...,pred_mean_mean,pred_mean_std,pred_max_mean,pred_max_std,voxel_auroc_mean,voxel_auroc_std,mlp_bpp_pct_improvement_mean,mlp_bpp_pct_improvement_std,mlp_batch_roc_auc_mean,mlp_batch_roc_auc_std
0,neurovault,cnn_ae_mixed,native_atlas_free_volume,16,16,0,NaN,0.008799,0.004802,0.008799,...,0.136502,0.080811,0.979847,0.061187,0.954150,0.025996,NaN,NaN,NaN,NaN
1,neurovault,cnn_ae_neurovault,native_atlas_free_volume,16,16,0,NaN,0.008347,0.004673,0.008347,...,0.131776,0.079768,0.976883,0.063121,0.952853,0.026035,NaN,NaN,NaN,NaN
2,neurovault,cnn_ae_nilearn,native_atlas_free_volume,16,16,0,NaN,0.008881,0.004844,0.008881,...,0.122440,0.073459,0.958895,0.073659,0.950300,0.025972,NaN,NaN,NaN,NaN
3,neurovault,cnn_ae_pubmed,native_atlas_free_volume,16,16,0,NaN,0.011826,0.006520,0.011826,...,0.114703,0.068463,0.980763,0.074505,0.941998,0.031630,NaN,NaN,NaN,NaN
4,neurovault,mlp_neurovlm,atlas_free_volume_via_mlp_masker_crop,16,16,0,NaN,0.308302,0.147779,0.308302,...,0.007042,0.005060,0.891323,0.077528,0.987513,0.009849,NaN,NaN,NaN,NaN
5,nilearn,cnn_ae_mixed,native_atlas_free_volume,16,16,0,NaN,0.000993,0.001501,0.000993,...,0.004496,0.006074,0.943071,0.214794,0.954289,0.101016,NaN,NaN,NaN,NaN
6,nilearn,cnn_ae_neurovault,native_atlas_free_volume,16,16,0,NaN,0.000972,0.001453,0.000972,...,0.004480,0.005687,0.938950,0.229528,0.947838,0.108476,NaN,NaN,NaN,NaN
7,nilearn,cnn_ae_nilearn,native_atlas_free_volume,16,16,0,NaN,0.000967,0.001462,0.000967,...,0.004262,0.005367,0.939875,0.232864,0.935786,0.113383,NaN,NaN,NaN,NaN
8,nilearn,cnn_ae_pubmed,native_atlas_free_volume,16,16,0,NaN,0.001186,0.001928,0.001186,...,0.004003,0.005484,0.938696,0.237430,0.950720,0.107167,NaN,NaN,NaN,NaN
9,nilearn,mlp_neurovlm,atlas_free_volume_via_mlp_masker_crop,16,16,0,NaN,0.003626,0.004704,0.003626,...,0.004890,0.005494,0.721555,0.324469,0.994037,0.009081,NaN,NaN,NaN,NaN


In [6]:
by_sample = pd.read_csv(result["by_sample_path"])
by_sample.head(20)

,dataset,model_id,model_family,comparison_space,supported,unsupported_reason,sample_index,map_id,source,source_detail,...,top5_overlap,top10_overlap,target_nonzero_fraction,pred_nonzero_fraction,pred_mean,pred_max,voxel_auroc,mlp_bpp_pct_improvement,mlp_batch_roc_auc,model_domain
0,pubmed,mlp_neurovlm,mlp,mlp_masker_flatmap,True,NaN,0,1589767,pubmed,pubmed,...,0.595238,0.695271,0.231799,1.000000,0.005433,0.603663,0.974012,36.993257,0.989851,NaN
1,pubmed,mlp_neurovlm,mlp,mlp_masker_flatmap,True,NaN,1,8530552,pubmed,pubmed,...,0.787815,0.753765,0.160956,1.000000,0.003495,0.750377,0.988818,61.179831,0.989851,NaN
2,pubmed,mlp_neurovlm,mlp,mlp_masker_flatmap,True,NaN,2,8624678,pubmed,pubmed,...,0.659664,0.649737,0.267606,1.000000,0.005551,0.969348,0.976171,50.989465,0.989851,NaN
3,pubmed,mlp_neurovlm,mlp,mlp_masker_flatmap,True,NaN,3,8670634,pubmed,pubmed,...,0.682773,0.660946,0.626095,1.000000,0.014230,0.789054,0.972025,16.756750,0.989851,NaN
4,pubmed,mlp_neurovlm,mlp,mlp_masker_flatmap,True,NaN,4,8994101,pubmed,pubmed,...,0.738796,0.620315,0.088431,1.000000,0.004282,0.934103,0.971198,65.581254,0.989851,NaN
5,pubmed,mlp_neurovlm,mlp,mlp_masker_flatmap,True,NaN,5,9038284,pubmed,pubmed,...,0.677871,0.697723,0.510125,1.000000,0.013628,0.628930,0.982772,20.950811,0.989851,NaN
6,pubmed,mlp_neurovlm,mlp,mlp_masker_flatmap,True,NaN,6,9065511,pubmed,pubmed,...,0.468487,0.607706,0.474108,1.000000,0.014967,0.529263,0.935706,12.147130,0.989851,NaN
7,pubmed,mlp_neurovlm,mlp,mlp_masker_flatmap,True,NaN,7,9084599,pubmed,pubmed,...,0.607143,0.692119,0.796405,1.000000,0.040672,0.831353,0.969853,19.385497,0.989851,NaN
8,pubmed,mlp_neurovlm,mlp,mlp_masker_flatmap,True,NaN,8,9106283,pubmed,pubmed,...,0.568627,0.585289,0.435253,1.000000,0.009564,0.814420,0.946986,38.389407,0.984424,NaN
9,pubmed,mlp_neurovlm,mlp,mlp_masker_flatmap,True,NaN,9,9114263,pubmed,pubmed,...,0.666667,0.775131,0.528940,1.000000,0.042913,0.931920,0.980891,26.715154,0.984424,NaN


## Visualize Results

Grouped bars compare mean metrics across models within each dataset. Color is
fixed per **model family** (MLP, CNN mixed baseline, CNN domain-specialized)
across every chart in this notebook and in the other two comparison
notebooks -- the same model always reads as the same color. Bars are
annotated with their value; a missing bar means that model/dataset
combination produced no supported rows (see the coverage table below for
why, e.g. `missing_checkpoint`).

The MLP autoencoder was trained on **binary** PubMed activation masks, so on
**PubMed** it is evaluated with the main package's dedicated flat resource
(`mlp_masker_flatmap`). NeuroVault and Nilearn have no such resource, but the
atlas-free CNN's packed `(36, 45, 38)` volumes turn out to be crops of the
*exact same MNI152 4mm grid* the MLP masker uses (see
`atlas_free_cnn.evaluation.mlp_masker_bridge`) -- so those volumes are
converted into MLP masker-flat space with a boolean crop index (no
resampling), binarized to match the MLP's training distribution, and
evaluated the same way as PubMed. This shows up as a second comparison
space, `atlas_free_volume_via_mlp_masker_crop`, in the coverage table and
plots below.


In [7]:
import matplotlib.pyplot as plt
import plotting_utils as pu

coverage_df = summary.copy()
if not coverage_df.empty:
    coverage_df["status"] = coverage_df["n_supported"].gt(0).map({True: "ok", False: "unsupported"})
pu.coverage_table(coverage_df)

dataset,neurovault,nilearn,pubmed
model_id,,,
cnn_ae_mixed,ok,ok,ok
cnn_ae_neurovault,ok,ok,ok
cnn_ae_nilearn,ok,ok,ok
cnn_ae_pubmed,ok,ok,ok
mlp_neurovlm,ok,ok,ok


In [9]:
REPORT_ASSETS_DIR = OUTPUT_DIR / "report_assets" / "ae_reconstruction"

report_manifest = pu.save_report_assets(
    REPORT_ASSETS_DIR,
    figures={"ae_reconstruction_metrics": fig_metrics},
    dataframes={"ae_reconstruction_summary": summary, "ae_reconstruction_by_sample": by_sample},
)
report_manifest

{'figures': {'ae_reconstruction_metrics': '/Users/borng/code/lab_work/neurovlm/experiments/3dcnn/atlas_free_cnn/outputs/model_comparison/report_assets/ae_reconstruction/ae_reconstruction_metrics.png'},
 'dataframes': {'ae_reconstruction_summary': '/Users/borng/code/lab_work/neurovlm/experiments/3dcnn/atlas_free_cnn/outputs/model_comparison/report_assets/ae_reconstruction/ae_reconstruction_summary.csv',
  'ae_reconstruction_by_sample': '/Users/borng/code/lab_work/neurovlm/experiments/3dcnn/atlas_free_cnn/outputs/model_comparison/report_assets/ae_reconstruction/ae_reconstruction_by_sample.csv'},
 'manifest_path': '/Users/borng/code/lab_work/neurovlm/experiments/3dcnn/atlas_free_cnn/outputs/model_comparison/report_assets/ae_reconstruction/manifest.json'}